# Week 24 · Notebook 1: Governance & FinOps

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `shipments.csv`
- `support_tickets.csv`

System tables (`system.access.audit`, `system.access.table_lineage`, `system.billing.usage`) must be enabled (they are by default in most workspaces). This notebook applies **row filters**, **column masks**, and a **dynamic view**, then reads **audit**, **lineage**, and **billing** data and ends with a cost metric.


## Governance & FinOps in one notebook

- **Row filters**: a boolean UDF applied with `SET ROW FILTER` so a group sees only allowed rows.
- **Column masks**: `ALTER COLUMN … SET MASK` hides a column's value for everyone but the allowed group.
- **Dynamic views**: a view that branches on `is_account_group_member()` to redact PII.
- **Audit / lineage**: `system.access.audit` and `system.access.table_lineage` / `column_lineage`.
- **FinOps**: `system.billing.usage` aggregated into a cost-by-day table, ending in a dollar metric.

See research §8, §9 and `reference/platforms/databricks/16-governance-security.md` + `17-finopps-cost.md`.


In [ ]:
# Ensure the governed tables exist. If week-21 did not run, recreate a typed silver table + tickets.
if not spark.catalog.tableExists("zrl_.zorologistics.silver_shipments"):
    spark.sql("""
      CREATE TABLE zrl_.zorologistics.silver_shipments AS
      SELECT
        shipment_id, carrier_id, lane_id, commodity,
        CAST(weight_kg AS DOUBLE) AS weight_kg,
        CAST(value_usd AS DOUBLE) AS value_usd,
        to_timestamp(planned_departure, 'yyyy-MM-dd HH:mm:ss[.SSSSSS]') AS planned_departure,
        CAST(delay_hours AS DOUBLE) AS delay_hours,
        status, weather_severity
      FROM read_files('/Volumes/zrl_/zorologistics/raw/shipments.csv',
                      format => 'csv', header => true, inferSchema => true)
    """)

if not spark.catalog.tableExists("zrl_.zorologistics.support_tickets"):
    spark.sql("""
      CREATE TABLE zrl_.zorologistics.support_tickets AS
      SELECT * FROM read_files('/Volumes/zrl_/zorologistics/raw/support_tickets.csv',
                               format => 'csv', header => true, inferSchema => true)
    """)

print("silver rows:", spark.sql("SELECT count(*) FROM zrl_.zorologistics.silver_shipments").collect()[0][0])
print("ticket rows:", spark.sql("SELECT count(*) FROM zrl_.zorologistics.support_tickets").collect()[0][0])


## Row filters

A **row filter** is a boolean UDF. `is_account_group_member('carrier_ops')` lets members of that group see everything; everyone else sees only the three listed carriers. Additive-only privilege model: the filter *restricts*, it never grants.


In [ ]:
%sql
-- Row filter: a boolean UDF restricting which rows a group can see.
CREATE OR REPLACE FUNCTION zrl_.zorologistics.filter_carrier(carrier_id STRING)
RETURN IF(is_account_group_member('carrier_ops'), TRUE,
          carrier_id IN ('C001', 'C002', 'C003'));

ALTER TABLE zrl_.zorologistics.silver_shipments
SET ROW FILTER zrl_.zorologistics.filter_carrier ON (carrier_id);


In [ ]:
%sql
-- The visible row count is now restricted for non-carrier_ops users.
SELECT count(*) AS visible_rows,
       count(DISTINCT carrier_id) AS visible_carriers
FROM zrl_.zorologistics.silver_shipments


## Column masks

A **column mask** rewrites a column's value by group. `is_account_group_member('finance')` sees `value_usd`; everyone else sees `NULL`. The function's return type must match the column type (`DOUBLE` here).


In [ ]:
%sql
-- Column mask: hide value_usd from everyone except the finance group.
CREATE OR REPLACE FUNCTION zrl_.zorologistics.mask_value(value DOUBLE)
RETURN IF(is_account_group_member('finance'), value, NULL);

ALTER TABLE zrl_.zorologistics.silver_shipments
ALTER COLUMN value_usd SET MASK zrl_.zorologistics.mask_value;


In [ ]:
%sql
-- Non-finance users now see NULL in value_usd; finance sees the real numbers.
SELECT shipment_id, carrier_id, value_usd
FROM zrl_.zorologistics.silver_shipments
LIMIT 5


## A dynamic view for customer PII

A **dynamic view** bakes the branch into SQL: `is_account_group_member('support_team')` shows the full ticket text; everyone else sees a redacted placeholder. This is the pattern for customer PII you never want to leave the warehouse.


In [ ]:
%sql
-- Dynamic view: full ticket text only for the support team; redacted for everyone else.
CREATE OR REPLACE VIEW zrl_.zorologistics.v_customer_tickets AS
SELECT
  ticket_id,
  shipment_id,
  customer_id,
  CASE WHEN is_account_group_member('support_team') THEN text
       ELSE '*** REDACTED ***' END AS text,
  category,
  priority
FROM zrl_.zorologistics.support_tickets


In [ ]:
%sql
-- Preview the dynamic view (a non-support user sees redacted text).
SELECT ticket_id, customer_id, text, category
FROM zrl_.zorologistics.v_customer_tickets
LIMIT 5


## Audit: who did what

`system.access.audit` records every action. This is the "who touched the table" query for a security review. Column names can vary by workspace, run `DESCRIBE system.access.audit` first if a name is unfamiliar.


In [ ]:
%sql
-- Audit: recent events, newest first.
SELECT event_time, action_name, user_identity.email, workspace_id
FROM system.access.audit
ORDER BY event_time DESC
LIMIT 20


## Lineage: what feeds what

`system.access.table_lineage` (tables) and `system.access.column_lineage` (columns) map data flow. We ask which source tables feed the gold KPI table.


In [ ]:
%sql
-- Lineage: which source tables feed the gold KPI table?
SELECT source_table_full_name, target_table_full_name
FROM system.access.table_lineage
WHERE target_table_full_name LIKE '%gold_on_time_kpis'


In [ ]:
%sql
-- Column lineage for the on_time_rate column (if populated).
SELECT source_table_full_name, source_column_name, target_table_full_name, target_column_name
FROM system.access.column_lineage
WHERE target_column_name = 'on_time_rate'


## FinOps: billing

`system.billing.usage` holds granular DBU records (`sku_name`, `usage_quantity`, `billing_origin_product`). We aggregate into a **cost-by-day** table, then estimate dollars with an illustrative DBU rate. On a brand-new trial this table may be empty for a while, the query still returns a valid (0) number.


In [ ]:
%sql
-- FinOps: DBUs consumed per day.
CREATE OR REPLACE TABLE zrl_.zorologistics.cost_by_day AS
SELECT
  usage_date AS usage_day,
  sum(usage_quantity) AS total_dbu,
  count(DISTINCT workspace_id) AS workspaces
FROM system.billing.usage
GROUP BY usage_date
ORDER BY usage_date


In [ ]:
%sql
-- The cost-by-day table.
SELECT * FROM zrl_.zorologistics.cost_by_day
ORDER BY usage_day DESC
LIMIT 10


In [ ]:
# Final metric: total DBUs, estimated cost, and audit-event volume.
total_dbu = spark.sql(
    "SELECT sum(total_dbu) FROM zrl_.zorologistics.cost_by_day").collect()[0][0]
total_dbu = total_dbu if total_dbu is not None else 0.0

rate = 0.55 # illustrative USD per DBU (actual rate varies by SKU/region, join list_prices for real cost)
est_cost = total_dbu * rate

audit_events = spark.sql("SELECT count(*) FROM system.access.audit").collect()[0][0]

print("total DBUs (billing):", total_dbu)
print("estimated cost USD:", round(est_cost, 2))
print("audit events:", audit_events)
